# Real-Time Distributed Ingestion: Kafka vs. Flume
### Part A
#### 3 — Kafka core concepts và tích hợp Spark Structured Streaming


- Định nghĩa **Broker, Topic, Partition, Producer, Consumer, Consumer Group**.
- Giải thích kiến trúc **ZooKeeper / KRaft**, ưu và nhược điểm của Kafka.
- So sánh Kafka với Flume.
- Giải thích kết nối Spark qua **spark-sql-kafka-0-10**.

**Dữ liệu minh họa: 180 sự kiện mô phỏng theo schema Retail Rocket, seed=42.**

Notebook tự sinh dữ liệu để giải thích cơ chế trong Part A.



## Cách chạy và mục lục

Chọn kernel **Python 3 có pandas**, rồi chạy các cell từ trên xuống dưới.
Notebook tự sinh dữ liệu trong bộ nhớ với **seed=42**, có thể chạy độc lập
với các file CSV đã tải.

| Mục | Nội dung | Minh họa |
|---|---|---|
| 1 | Dữ liệu theo schema Retail Rocket | Sinh 180 sự kiện có thể tái lập |
| 2 | Khái niệm Kafka | Ánh xạ vào pipeline của nhóm |
| 3 | Producer | Chuyển sự kiện thành key và JSON bytes |
| 4 | Partition | Phân vùng theo visitorid và kiểm tra thứ tự |
| 5 | Consumer Group | Chia partition và đọc độc lập giữa các group |
| 6 | Offset và replay | Lưu tiến độ, đọc tiếp và đọc lại |
| 7 | ZooKeeper / KRaft | Metadata, broker và controller |
| 8 | Ưu và nhược điểm | Liên hệ với clickstream |
| 9 | Kafka vs. Flume | Bảng so sánh và ví dụ channel / retained log |
| 10 | Spark Kafka connector | Từ Kafka bytes đến DataFrame có schema |
| 11 | Code minh họa cho Part B | Hàm đọc Kafka và parse JSON bằng Spark |

**Phạm vi kết quả:** các cell mô phỏng cơ chế Kafka/Flume bằng Python;
chúng không khởi động broker, agent Flume hoặc Spark.
Mục 11 định nghĩa một hàm sử dụng API Spark, để giải thích cách tích hợp ở Part B.
Hàm đó được trình bày trong notebook nhưng chưa được gọi.

## 1. Tạo dữ liệu mô phỏng theo schema Retail Rocket

| Trường | Ý nghĩa |
|---|---|
| timestamp | Thời điểm sự kiện, UNIX milliseconds |
| visitorid | Mã người truy cập |
| event | view, addtocart hoặc transaction |
| itemid | Mã sản phẩm |
| transactionid | Mã giao dịch; null với view và addtocart |

Ta tự sinh **180 sự kiện**, dùng **12 visitor** và mã sản phẩm **1000–1008**.
Timestamp mô phỏng tăng 15 giây cho mỗi sự kiện; chương trình không chờ 15 giây
ngoài thực tế.

Xác suất chọn hành động được đặt ở mức **view 80%, addtocart 15%, transaction 5%**.
Đây là tham số tự chọn để dễ minh họa, không phải thống kê từ dataset thật.
Số lượng thực tế của từng hành động trong mẫu ngẫu nhiên có thể khác các tỷ lệ này.

Dùng bộ sinh ngẫu nhiên riêng với seed=42 giúp tái tạo đúng cùng một mẫu.
Không ghi file trung gian; dữ liệu tồn tại trong bộ nhớ để notebook chạy gọn.

In [1]:
from collections import deque
import hashlib
import json
import random
import sys
import pandas as pd
from IPython.display import display

FIELDS = ["timestamp", "visitorid", "event", "itemid", "transactionid"]
N_EVENTS = 180
SEED = 42
EPOCH0 = 1433221332117
TOPIC = "retail-rocket-events"
N_PARTITIONS = 2

pd.set_option("display.max_columns", 10)
pd.set_option("display.max_colwidth", 55)
print("Python:", sys.version.split()[0])
print("Nguồn: dữ liệu mô phỏng theo schema Retail Rocket")
print("Số sự kiện:", N_EVENTS, "| Seed:", SEED)

Python: 3.12.4
Nguồn: dữ liệu mô phỏng theo schema Retail Rocket
Số sự kiện: 180 | Seed: 42


In [2]:
def generate_events(n=N_EVENTS, seed=SEED):
    rng = random.Random(seed)
    rows = []
    for i in range(n):
        action = rng.choices(
            ["view", "addtocart", "transaction"],
            weights=[80, 15, 5],
        )[0]
        rows.append({
            "timestamp": EPOCH0 + i * 15_000,
            "visitorid": rng.randint(1, 12),
            "event": action,
            "itemid": rng.randint(1000, 1008),
            "transactionid": (
                rng.randint(1, 9999) if action == "transaction" else None
            ),
        })
    return rows

synthetic_events = generate_events()
df = pd.DataFrame(synthetic_events).astype({
    "timestamp": "int64",
    "visitorid": "int64",
    "event": "string",
    "itemid": "int64",
    "transactionid": "Int64",
})

assert len(df) == N_EVENTS
assert list(df.columns) == FIELDS
assert synthetic_events == generate_events()
assert (df["transactionid"].notna() == df["event"].eq("transaction")).all()

print(f"Đã sinh {len(df)} sự kiện; {df['visitorid'].nunique()} visitor.")
display(df["event"].value_counts().rename_axis("event").to_frame("records"))
display(df.groupby("event", sort=False).head(2).reset_index(drop=True))

Đã sinh 180 sự kiện; 12 visitor.


,records
event,
view,151
addtocart,22
transaction,7


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,1,view,1004,<NA>
1,1433221347117,3,view,1001,<NA>
2,1433221452117,1,addtocart,1002,<NA>
3,1433221512117,10,addtocart,1004,<NA>
4,1433221602117,4,transaction,1001,6228
5,1433221992117,3,transaction,1008,1744


**Đọc kết quả:** bảng đầu cho biết số view, addtocart và transaction của mẫu
vừa sinh. Bảng sau cho thấy transactionid được giữ là null khi chưa mua.
Chạy lại với cùng seed cho cùng mẫu; đổi seed để quan sát kết quả khác.

Mẫu này phục vụ giải thích kiến trúc, không dùng để kết luận về hành vi mua sắm
thực tế hoặc hiệu năng của hệ thống.

## 2. Các khái niệm cốt lõi của Kafka

Kafka lưu sự kiện trong một **distributed log**: record được nối thêm vào cuối
partition và giữ theo chính sách retention.

| Khái niệm | Định nghĩa | Trong pipeline Part B |
|---|---|---|
| Broker | Server lưu partition, phục vụ yêu cầu đọc và ghi. | Kafka broker của pipeline. |
| Topic | Luồng record có tên; đọc không xóa record ngay. | retail-rocket-events |
| Partition | Một log có thứ tự trong topic, dùng phân phối dữ liệu. | Topic cần ít nhất hai partition. |
| Producer | Ứng dụng gửi key, value và metadata vào Kafka. | Python đọc CSV rồi gửi JSON. |
| Consumer | Ứng dụng đọc record và theo dõi vị trí trong partition. | Spark là ứng dụng xử lý phía sau Kafka. |
| Consumer Group | Nhóm consumer chia partition; các group đọc độc lập. | Hệ gợi ý và hệ giám sát có thể có group riêng. |
| Offset | Số thứ tự của record trong một partition. | Xác định dữ liệu cần đọc tiếp hoặc replay. |

**Broker khác partition:** một broker có thể lưu nhiều partition.
Hai partition trên một broker vẫn có một điểm lỗi chung là broker đó.

## 3. Producer — từ sự kiện mô phỏng đến Kafka record

- **Key:** visitorid, dùng để gom hành vi liên quan vào cùng partition.
- **Value:** JSON chứa đúng năm trường của Retail Rocket.
- transactionid thiếu phải trở thành **JSON null**, không phải NaN.

Cell sau tạo payload bytes từ dữ liệu mô phỏng.
Trong Part B, producer sẽ đọc sự kiện từ CSV thật rồi thực hiện bước tuần tự hóa
tương tự trước khi gửi tới Kafka.

In [3]:
def to_event(row):
    return {
        "timestamp": int(row.timestamp),
        "visitorid": int(row.visitorid),
        "event": str(row.event),
        "itemid": int(row.itemid),
        "transactionid": (
            None if pd.isna(row.transactionid) else int(row.transactionid)
        ),
    }

events = [to_event(row) for row in df.itertuples(index=False)]
encoded = [
    {
        "key": str(event["visitorid"]).encode("utf-8"),
        "value": json.dumps(
            event, ensure_ascii=False, allow_nan=False, separators=(",", ":")
        ).encode("utf-8"),
    }
    for event in events
]
print("Key của bản ghi đầu:", encoded[0]["key"])
print("Value bytes:", encoded[0]["value"])
print("\nVí dụ từ từng loại hành động trong dữ liệu mô phỏng:")
for action in ("view", "addtocart", "transaction"):
    example = next((e for e in events if e["event"] == action), None)
    if example is not None:
        print(json.dumps(example, ensure_ascii=False, indent=2))

Key của bản ghi đầu: b'1'
Value bytes: b'{"timestamp":1433221332117,"visitorid":1,"event":"view","itemid":1004,"transactionid":null}'

Ví dụ từ từng loại hành động trong dữ liệu mô phỏng:
{
  "timestamp": 1433221332117,
  "visitorid": 1,
  "event": "view",
  "itemid": 1004,
  "transactionid": null
}
{
  "timestamp": 1433221452117,
  "visitorid": 1,
  "event": "addtocart",
  "itemid": 1002,
  "transactionid": null
}
{
  "timestamp": 1433221602117,
  "visitorid": 4,
  "event": "transaction",
  "itemid": 1001,
  "transactionid": 6228
}


## 4. Partition — chia tải và giữ thứ tự

Kafka đảm bảo thứ tự **trong partition**, theo thứ tự ghi vào log.
Kafka không tự sắp xếp theo timestamp và không tạo thứ tự chung cho cả topic.

Ta dùng hash cố định trên visitorid để chọn một trong hai partition.
**SHA-256 chỉ là hàm định tuyến minh họa**, không tái hiện partitioner mặc định
của Kafka client. Client thực tế quyết định partition.

Mỗi partition có offset riêng: offset 0 ở P0 khác offset 0 ở P1.

In [4]:
def demo_partition(key, partition_count=N_PARTITIONS):
    digest = hashlib.sha256(key).digest()
    return int.from_bytes(digest[:8], "big") % partition_count

logs = {p: [] for p in range(N_PARTITIONS)}
for source_row, (event, payload) in enumerate(zip(events, encoded)):
    partition = demo_partition(payload["key"])
    logs[partition].append({
        "topic": TOPIC, "partition": partition,
        "offset": len(logs[partition]), "source_row": source_row,
        "visitorid": event["visitorid"], "event": event["event"],
        "key": payload["key"], "value": payload["value"],
    })

meta = pd.DataFrame([
    {k: r[k] for k in ("partition", "offset", "source_row", "visitorid", "event")}
    for records in logs.values() for r in records
])
display(meta.groupby("partition").agg(
    records=("offset", "size"), visitors=("visitorid", "nunique"),
    first_offset=("offset", "min"), last_offset=("offset", "max"),
))
assert meta.groupby("visitorid")["partition"].nunique().eq(1).all()
for records in logs.values():
    assert [r["offset"] for r in records] == list(range(len(records)))
    assert [r["source_row"] for r in records] == sorted(
        r["source_row"] for r in records
    )

visitor = df["visitorid"].value_counts().index[0]
print("Một visitor có nhiều hành động trong mẫu:", visitor)
display(
    meta[meta["visitorid"].eq(visitor)]
    .sort_values("offset").head(8).reset_index(drop=True)
)

,records,visitors,first_offset,last_offset
partition,,,,
0,118,7,0,117
1,62,5,0,61


Một visitor có nhiều hành động trong mẫu: 9


,partition,offset,source_row,visitorid,event
0,0,0,2,9,view
1,0,2,5,9,view
2,0,18,32,9,view
3,0,21,39,9,view
4,0,26,47,9,view
5,0,30,53,9,transaction
6,0,31,54,9,view
7,0,33,56,9,transaction


**Đọc kết quả:** visitor được chọn nằm trong một partition với offset tăng dần.
Các kiểm tra xác nhận logic mô phỏng; đây chưa phải phép đo throughput Kafka.

Nếu đổi số partition hoặc cách định tuyến, ánh xạ key → partition có thể thay đổi.
Hai partition tạo khả năng đọc song song, còn mức song song thực tế phụ thuộc
tài nguyên và cách ứng dụng đọc dữ liệu.

## 5. Consumer và Consumer Group

Trong một **consumer group thông thường**, một partition được gán cho tối đa
một consumer của group tại một thời điểm; một consumer có thể nhận nhiều partition.

- Hai partition, hai consumer: mỗi consumer có thể nhận một partition.
- Hai partition, ba consumer: có ít nhất một consumer không được gán partition.
- Hai group khác nhau có tiến độ riêng, nên đều có thể đọc cả topic.

Hàm dưới chia round-robin để minh họa nguyên tắc, không thực hiện rebalance Kafka.

In [5]:
def demo_assignment(group, members):
    assignment = {member: [] for member in members}
    for partition in range(N_PARTITIONS):
        assignment[members[partition % len(members)]].append(partition)
    return [
        {
            "group": group, "consumer": member, "partitions": parts,
            "records_available": sum(len(logs[p]) for p in parts),
        }
        for member, parts in assignment.items()
    ]

group_a = demo_assignment("recommendations", ["consumer-A1", "consumer-A2"])
group_b = demo_assignment(
    "monitoring", ["consumer-B1", "consumer-B2", "consumer-B3"]
)
display(pd.DataFrame(group_a + group_b))
for assignment in (group_a, group_b):
    assigned = [p for c in assignment for p in c["partitions"]]
    assert sorted(assigned) == list(range(N_PARTITIONS))
    assert sum(c["records_available"] for c in assignment) == len(events)
print("Mỗi group có thể đọc đủ", len(events), "record với tiến độ độc lập.")

,group,consumer,partitions,records_available
0,recommendations,consumer-A1,[0],118
1,recommendations,consumer-A2,[1],62
2,monitoring,consumer-B1,[0],118
3,monitoring,consumer-B2,[1],62
4,monitoring,consumer-B3,[],0


Mỗi group có thể đọc đủ 180 record với tiến độ độc lập.


## 6. Offset, lưu tiến độ và replay

Offset là vị trí trong **một partition**, không phải visitorid hay timestamp.
Sau khi đọc các offset 0–4, vị trí cần đọc tiếp là 5.

Record vẫn được lưu sau khi consumer đọc, cho đến khi chính sách retention
hoặc compaction cho phép loại bỏ nó. Consumer có thể đọc lại record còn tồn tại.

Cell sau minh họa đọc lần đầu, đọc tiếp và replay từ đầu. Dict lưu tiến độ
trong ví dụ chưa phải checkpoint Spark.

In [6]:
def read_demo_batch(positions, per_partition=3):
    batch, next_positions = [], dict(positions)
    for partition, records in logs.items():
        start = positions.get(partition, 0)
        selected = records[start:start + per_partition]
        batch.extend(selected)
        next_positions[partition] = start + len(selected)
    return batch, next_positions

start = {p: 0 for p in logs}
batch_1, saved_positions = read_demo_batch(start)
batch_2, resumed_positions = read_demo_batch(saved_positions)
replayed, _ = read_demo_batch(start)

def ids(batch):
    return {(r["partition"], r["offset"]) for r in batch}

assert ids(batch_1).isdisjoint(ids(batch_2))
assert ids(batch_1) == ids(replayed)
assert sum(map(len, logs.values())) == len(events)
display(pd.DataFrame([
    {"step": "Đọc đầu", "records": len(batch_1), "next_offsets": saved_positions},
    {"step": "Đọc tiếp", "records": len(batch_2), "next_offsets": resumed_positions},
    {"step": "Replay", "records": len(replayed), "next_offsets": saved_positions},
]))
print("Đọc tiếp không lặp batch đầu; replay trả lại cùng partition/offset.")
print("Log vẫn giữ đủ", sum(map(len, logs.values())), "record sau khi đọc.")

,step,records,next_offsets
0,Đọc đầu,6,"{0: 3, 1: 3}"
1,Đọc tiếp,6,"{0: 6, 1: 6}"
2,Replay,6,"{0: 3, 1: 3}"


Đọc tiếp không lặp batch đầu; replay trả lại cùng partition/offset.
Log vẫn giữ đủ 180 record sau khi đọc.


## 7. ZooKeeper và KRaft architecture

**ZooKeeper mode, trước đây:** Kafka dùng ZooKeeper riêng để phối hợp metadata
cluster như thông tin broker và quản lý controller.

**KRaft mode:** Kafka dùng quorum controller theo Raft để quản lý metadata.
Broker xử lý dữ liệu, controller quản lý metadata và phối hợp cluster.
Raft ở đây áp dụng cho metadata; clickstream không đi qua controller như một
bước trung gian trong đường gửi record.

<pre>
Python producer → Broker: topic / partitions → Consumer / Spark
                              ↑
                    metadata và điều phối
                              ↑
                    KRaft controller quorum
</pre>

Kafka 4.x đã bỏ ZooKeeper mode. Compose hiện tại dự kiến Kafka 4.3.1 với
process.roles = broker,controller cho demo local.

**Giới hạn:** một broker với replication factor 1 không chứng minh chịu lỗi broker.
Hai partition không đồng nghĩa hai bản sao. Các cell mô phỏng cũng không kiểm
chứng quorum hoặc failover thật.

## 8. Ưu và nhược điểm của Kafka

| Ưu điểm | Ý nghĩa với Retail Rocket |
|---|---|
| Tách producer khỏi consumer | Producer gửi sự kiện; Spark đọc theo tốc độ của nó trong giới hạn dung lượng lưu trữ. |
| Retention và replay | Tính lại intent từ các offset vẫn được giữ. |
| Nhiều ứng dụng đọc độc lập | Hệ gợi ý và giám sát dùng chung nguồn clickstream. |
| Phân vùng để mở rộng | Phân phối dữ liệu và công việc đọc qua nhiều partition. |
| Replication khi đủ broker | Hỗ trợ chịu lỗi nếu triển khai và cấu hình phù hợp. |

| Nhược điểm / giới hạn | Liên hệ dự án |
|---|---|
| Phải vận hành broker, disk, retention, controller | Demo local đơn giản hơn cluster production. |
| Chỉ giữ thứ tự trong partition | Chọn key phù hợp, xử lý event-time ở Spark. |
| Key có thể gây lệch tải | Visitor hoạt động nhiều có thể tạo partition nóng. |
| Offset hết retention không replay được | Checkpoint không thể lấy lại record đã bị xóa. |
| Kafka không tự tính intent hay làm sạch JSON | Spark và producer vẫn cần kiểm tra schema và transform. |

Độ trễ và throughput phụ thuộc cấu hình, phần cứng và workload.
Các output trong notebook chưa phải benchmark so sánh Kafka với Flume.

## 9. Kafka vs. Flume

Flume tổ chức luồng theo **source → channel → sink**. Source nhận sự kiện,
channel giữ sự kiện chờ chuyển tiếp, sink gửi tới đích hoặc agent kế tiếp.

File Channel lưu trên đĩa. Memory Channel mất sự kiện còn trong RAM nếu agent
dừng bất thường. Channel transaction hỗ trợ chuyển sự kiện qua từng hop;
độ tin cậy còn phụ thuộc source/sink, dung lượng và topology.

| Tiêu chí | Kafka | Flume |
|---|---|---|
| Vai trò chính | Lưu và phân phối event stream. | Thu thập và vận chuyển sự kiện theo source/channel/sink. |
| Lưu sau khi đọc | Record còn theo retention/compaction. | Sink lấy record khỏi channel khi chuyển tiếp thành công. |
| Replay độc lập | Đọc theo offset còn trong topic. | Channel thông thường không là retained log cho nhiều consumer độc lập. |
| Nhiều đích sử dụng | Consumer group có tiến độ độc lập. | Fan-out qua cấu hình channel/sink. |
| Chịu lỗi | Phụ thuộc replicas, broker và client. | Phụ thuộc channel, transaction và topology. |
| Tích hợp | Có source Kafka trực tiếp cho Spark Structured Streaming. | Có thể gửi log vào HDFS hoặc Kafka. |
| Trong bài này | Ingestion của pipeline clickstream. | Phương án để so sánh kiến trúc và trường hợp sử dụng. |

Có thể kết hợp **Flume → Kafka → Spark**. Điều này không có nghĩa mọi phiên bản
của các thành phần đã được kiểm thử tương thích với nhau.

In [7]:
# Minh họa logic lấy khỏi channel và đọc retained log.
# Không mô phỏng transaction hoặc độ bền Flume File Channel.
demo_records = encoded[:3]
channel = deque(demo_records)
channel_before = len(channel)
delivered = channel.popleft()  # giả sử sink đã chuyển tiếp thành công
retained_log = list(demo_records)
consumer_view = retained_log[:1]  # đọc theo offset, không xóa log

display(pd.DataFrame([
    {"model": "Channel: chuyển tiếp 1 event",
     "before": channel_before, "remaining": len(channel)},
    {"model": "Retained log: đọc 1 event",
     "before": len(demo_records), "remaining": len(retained_log)},
]))
print("Event mô phỏng dùng trong ví dụ:")
print(json.loads(delivered["value"].decode("utf-8")))

,model,before,remaining
0,Channel: chuyển tiếp 1 event,3,2
1,Retained log: đọc 1 event,3,3


Event mô phỏng dùng trong ví dụ:
{'timestamp': 1433221332117, 'visitorid': 1, 'event': 'view', 'itemid': 1004, 'transactionid': None}


## 10. Tích hợp Spark qua spark-sql-kafka-0-10

Connector là thư viện JVM giúp Spark Structured Streaming đọc Kafka thành
streaming DataFrame. Thư viện Python kafka-python phục vụ producer;
nó không thay thế connector phía Spark.

Với Spark 4.2.0 trong Compose, package là:

<pre>org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0</pre>

- **spark-sql-kafka-0-10:** hỗ trợ Kafka 0.10 trở lên, không buộc broker phải là 0.10.
- **_2.13:** phiên bản nhị phân Scala.
- **4.2.0:** phiên bản connector cần khớp Spark runtime.

Nạp package bằng spark-submit --packages hoặc cấu hình spark.jars.packages.

### Cơ chế đọc

<pre>
Kafka record
 → readStream.format("kafka")
 → binary key/value + topic, partition, offset, Kafka timestamp
 → CAST(value AS STRING)
 → from_json(value, schema)
 → các cột Retail Rocket có kiểu dữ liệu
 → timestamp_millis(timestamp) thành event_time
 → xử lý cửa sổ, intent và ghi sink
</pre>

Timestamp metadata của Kafka khác trường timestamp trong JSON.
Bài này lấy timestamp trong payload làm event_time.

In [8]:
# Kiểm tra payload mô phỏng bằng Python trước khi đưa vào from_json của Spark.
# Cell này không chạy Spark và không kết nối broker.
decoded = [json.loads(p["value"].decode("utf-8")) for p in encoded[:5]]
assert all(set(event) == set(FIELDS) for event in decoded)
assert all(isinstance(event["timestamp"], int) for event in decoded)
parsed_demo = pd.DataFrame(decoded)
parsed_demo["transactionid"] = pd.array(
    parsed_demo["transactionid"], dtype="Int64"
)
parsed_demo["event_time"] = pd.to_datetime(
    parsed_demo["timestamp"], unit="ms", utc=True
)
display(parsed_demo)
print("Mục 11 định nghĩa hàm dùng from_json của Spark cho kiểu payload này.")

,timestamp,visitorid,event,itemid,transactionid,event_time
0,1433221332117,1,view,1004,<NA>,2015-06-02 05:02:12.117000+00:00
1,1433221347117,3,view,1001,<NA>,2015-06-02 05:02:27.117000+00:00
2,1433221362117,9,view,1001,<NA>,2015-06-02 05:02:42.117000+00:00
3,1433221377117,1,view,1000,<NA>,2015-06-02 05:02:57.117000+00:00
4,1433221392117,4,view,1008,<NA>,2015-06-02 05:03:12.117000+00:00


Mục 11 định nghĩa hàm dùng from_json của Spark cho kiểu payload này.


### Offset và checkpoint trong Spark

Driver xác định khoảng offset cho mỗi micro-batch; các task đọc dữ liệu Kafka.
Spark quản lý tiến độ bằng checkpoint, thay vì dựa vào consumer auto-commit.

- startingOffsets="earliest" áp dụng khi **bắt đầu query mới**.
- Restart với cùng checkpoint tiếp tục từ tiến độ đã lưu.
- Checkpoint không thay retention: Kafka xóa offset cần đọc thì query có thể lỗi.
- End-to-end exactly-once còn cần sink có cơ chế ghi phù hợp.
  Console hoặc foreachBatch tùy ý không tự động bảo đảm exactly-once.

Mục 5 minh họa consumer group Kafka thông thường. Không suy ra cứng số task Spark
từ số consumer trong bảng đó: connector có cách phân chia công việc và cấu hình
như minPartitions riêng.

## 11. Code minh họa kết nối Spark để dùng ở Part B

Hàm dưới trình bày cơ chế connector: **đọc Kafka → chuyển binary thành
string → parse JSON theo schema → đổi timestamp thành event_time**.

Cell chỉ **định nghĩa hàm**. Các import PySpark nằm bên trong hàm và chỉ chạy
khi gọi hàm, nên Run All của notebook Part A vẫn chạy bằng Python/pandas.

Khi triển khai Part B, nhóm sẽ chuẩn bị Kafka đang chạy và SparkSession đã nạp
connector đúng phiên bản, sau đó truyền SparkSession vào hàm.
Địa chỉ broker là localhost:9092 nếu ứng dụng chạy trên host, hoặc kafka:19092
nếu ứng dụng ở cùng mạng Compose.

Phần B sẽ dùng các sự kiện đọc từ **events.csv thật**; schema và các bước parse
được giữ nhất quán với ví dụ mô phỏng này.

In [9]:
def build_kafka_stream(
    spark,
    bootstrap_servers="kafka:19092",
    topic=TOPIC,
):
    # Chỉ import và kết nối khi hàm được gọi trong môi trường Part B.
    from pyspark.sql import functions as F
    from pyspark.sql.types import (
        StructType, StructField, LongType, StringType
    )

    event_schema = StructType([
        StructField("timestamp", LongType(), True),
        StructField("visitorid", LongType(), True),
        StructField("event", StringType(), True),
        StructField("itemid", LongType(), True),
        StructField("transactionid", LongType(), True),
    ])

    raw_stream = (
        spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", bootstrap_servers)
        .option("subscribe", topic)
        .option("startingOffsets", "earliest")
        .option("failOnDataLoss", "true")
        .load()
    )
    return (
        raw_stream.select(
            "partition",
            "offset",
            F.from_json(
                F.col("value").cast("string"), event_schema
            ).alias("payload"),
        )
        .select("partition", "offset", "payload.*")
        .withColumn("event_time", F.timestamp_millis("timestamp"))
    )

print("Đã định nghĩa build_kafka_stream để minh họa code tích hợp.")
print("Hàm chưa được gọi; cell này không khởi động Spark hoặc kết nối Kafka.")

Đã định nghĩa build_kafka_stream để minh họa code tích hợp.
Hàm chưa được gọi; cell này không khởi động Spark hoặc kết nối Kafka.


**Giải thích các lệnh chính**

| Lệnh / tham số | Vai trò |
|---|---|
| spark | SparkSession được chuẩn bị khi triển khai Part B. |
| kafka.bootstrap.servers | Địa chỉ broker mà ứng dụng truy cập được. |
| subscribe | Topic chứa sự kiện. |
| readStream.format("kafka") | Tạo nguồn Kafka cho Structured Streaming. |
| CAST(value AS STRING) | Đổi payload binary thành chuỗi JSON. |
| from_json(..., event_schema) | Parse thành cột có kiểu dữ liệu. |
| timestamp_millis | Đổi UNIX milliseconds thành event_time. |
| partition, offset | Metadata xác định vị trí record trong topic. |

Khi gọi hàm, kết quả là streaming DataFrame; query bắt đầu xử lý khi gọi
writeStream.start ở phía sau. Các bước xử lý intent, ghi sink và thiết lập
checkpoint sẽ được triển khai trong Part B.

**Output của cell trên chỉ xác nhận đã định nghĩa hàm**, không phải kết quả
chạy Kafka–Spark. Các kết quả có dữ liệu trong notebook là minh họa Python
trên 180 sự kiện mô phỏng.

## 12. Tổng kết 

| Yêu cầu | Nơi trả lời |
|---|---|
| Broker, Topic, Partition, Producer, Consumer, Consumer Group | Mục 2–5 |
| Offset, lưu tiến độ, replay | Mục 6 |
| ZooKeeper / KRaft | Mục 7 |
| Ưu và nhược điểm Kafka | Mục 8 |
| Kafka vs. Flume | Mục 9 |
| Cơ chế spark-sql-kafka-0-10 | Mục 10–11 |

**Summary:**

Trong Part A, nhóm sinh 180 sự kiện đúng schema Retail Rocket để minh họa.
Producer biến mỗi sự kiện thành JSON, dùng visitorid làm key để gom hành vi
liên quan vào một partition. Consumer group giúp chia công việc và cho nhiều
ứng dụng đọc độc lập. Offset dùng theo dõi tiến độ và replay dữ liệu còn được giữ.
Kafka 4.x dùng KRaft quản lý metadata. Spark dùng spark-sql-kafka-0-10
để đọc Kafka và parse JSON thành DataFrame. Flume thu thập và vận chuyển log
theo source/channel/sink, và cũng có thể đưa dữ liệu vào Kafka.

**Kết quả lưu trong notebook:** mẫu dữ liệu mô phỏng, payload JSON, phân vùng,
consumer group và replay được minh họa bằng Python.
